## CREATING SCHEMA

In [ ]:


CREATE SCHEMA analysis;


## CREATING TABLE

In [ ]:
CREATE OR REPLACE TABLE ANALYsis.transactions (
    transaction_time NUMBER,
    amount FLOAT,
    is_fraud INTEGER,
    v1 FLOAT, v2 FLOAT, v3 FLOAT, v4 FLOAT, v5 FLOAT,
    v6 FLOAT, v7 FLOAT, v8 FLOAT, v9 FLOAT, v10 FLOAT,
    v11 FLOAT, v12 FLOAT, v13 FLOAT, v14 FLOAT, v15 FLOAT,
    v16 FLOAT, v17 FLOAT, v18 FLOAT, v19 FLOAT, v20 FLOAT,
    v21 FLOAT, v22 FLOAT, v23 FLOAT, v24 FLOAT, v25 FLOAT,
    v26 FLOAT, v27 FLOAT, v28 FLOAT
);

In [ ]:
CREATE OR REPLACE STAGE risk_analytics_stage;


In [ ]:
SELECT COUNT(*) FROM analysis.transactions;
SELECT * FROM analysis.transactions LIMIT 10;



In [ ]:
TRUNCATE TABLE analytics.transactions;


## POPULATING TABLE WITH DATA


In [ ]:
CREATE OR REPLACE FILE FORMAT fraud_csv_format
TYPE = CSV
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
SKIP_HEADER = 1
NULL_IF = ('', 'NULL');


In [ ]:
COPY INTO analysis.transactions
(
 transaction_time,
 v1, v2, v3, v4, v5, v6, v7, v8, v9, v10,
 v11, v12, v13, v14, v15, v16, v17, v18, v19, v20,
 v21, v22, v23, v24, v25, v26, v27, v28,
 amount,
 is_fraud
)
FROM (
    SELECT
        $1::NUMBER,        -- Time

        -- V1–V28 (28 columns)
        $2::FLOAT,  $3::FLOAT,  $4::FLOAT,  $5::FLOAT,
        $6::FLOAT,  $7::FLOAT,  $8::FLOAT,  $9::FLOAT,
        $10::FLOAT, $11::FLOAT, $12::FLOAT, $13::FLOAT,
        $14::FLOAT, $15::FLOAT, $16::FLOAT, $17::FLOAT,
        $18::FLOAT, $19::FLOAT, $20::FLOAT, $21::FLOAT,
        $22::FLOAT, $23::FLOAT, $24::FLOAT, $25::FLOAT,
        $26::FLOAT, $27::FLOAT, $28::FLOAT, $29::FLOAT,

        -- Amount + Class
        $30::FLOAT,        -- Amount
        $31::NUMBER        -- Class (is_fraud)

    FROM @risk_analytics_stage/CreditcardFraud.csv
)
FILE_FORMAT = fraud_csv_format
ON_ERROR = 'ABORT_STATEMENT';


## EDA

In [ ]:
SELECT
    transaction_time,
    amount,
    is_fraud
FROM analysis.transactions
LIMIT 10;



In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
# Load Snowflake table into Snowpark DataFrame
sf_df = session.table("ANALYSIS.TRANSACTIONS")

# Preview schema
sf_df.schema


In [ ]:
df = sf_df.to_pandas()

df.head()



In [ ]:
df.info()


In [ ]:
print("Missing values:", df.isnull().values.any())


In [ ]:
df['IS_FRAUD'].value_counts(normalize=True)


Dataset contains 284k transactions

Fraud rate 0.17% (extreme imbalance)

No missing values in key fields

In [ ]:
df['HOUR_OF_DAY'] = (df['TRANSACTION_TIME'] // 3600) % 24
df['DAY_INDEX'] = (df['TRANSACTION_TIME'] // (3600 * 24)).astype(int)


Transaction Amount Distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['AMOUNT'], bins=50)
plt.yscale('log')
plt.title("Transaction Amount Distribution (Log Scale)")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.show()

Transaction amounts are heavily right-skewed, with most activity concentrated in low-value transactions. A small subset of high-value transactions behave as outliers and may carry elevated fraud risk, motivating closer monitoring in downstream analysis.


Fraud Rate by Hour of Day

In [ ]:
hourly_fraud = df.groupby('HOUR_OF_DAY')['IS_FRAUD'].mean()

hourly_fraud.plot(kind='bar', figsize=(10,4))
plt.title("Fraud Rate by Hour of Day")
plt.ylabel("Fraud Rate")
plt.xlabel("Hour of Day")
plt.show()


Fraud risk is elevated during late-night and early-morning hours, with the highest fraud rates occurring between approximately 2–4 AM. This time-based pattern suggests that temporal signals may be useful for identifying higher-risk transactions in downstream analysis.


Fraud Rate by Amount Bucket

In [ ]:
df['AMOUNT_BUCKET'] = pd.cut(
    df['AMOUNT'],
    bins=[0, 50, 100, 500, 1000, 5000, 50000]
)

amount_fraud = df.groupby('AMOUNT_BUCKET')['IS_FRAUD'].mean()

amount_fraud.plot(kind='bar', figsize=(10,4))
plt.title("Fraud Rate by Transaction Amount")
plt.ylabel("Fraud Rate")
plt.xlabel("Amount Bucket")
plt.show()


Fraud rates increase as transaction amounts move into higher spend buckets, indicating that risk is unevenly distributed across transaction values. While not strictly monotonic, higher-value transactions exhibit materially higher fraud rates, supporting transaction amount as a meaningful risk signal in downstream modeling.


Time Behavior

In [ ]:
plt.figure(figsize=(10,4))
sns.histplot(df['HOUR_OF_DAY'], bins=24)
plt.title("Transaction Volume by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Transaction Count")
plt.show()


Transaction activity exhibits a clear daily pattern, with significantly lower volume during overnight hours and higher, more stable volume throughout the day. Deviations from these baseline temporal patterns may signal abnormal or higher-risk transaction behavior.

In [ ]:
df['high_value_flag'] = df['AMOUNT'] > 1000
df['night_txn_flag'] = df['HOUR_OF_DAY'].between(0, 5)

df['risk_score'] = (
    df['high_value_flag'].astype(int) * 2 +
    df['night_txn_flag'].astype(int)
)

df['risk_tier'] = pd.cut(
    df['risk_score'],
    bins=[-1, 1, 3, 10],
    labels=['LOW', 'MEDIUM', 'HIGH']
)



In [ ]:
risk_summary = (
    df.groupby('risk_tier')
      .agg(
          transactions=('AMOUNT', 'count'),
          frauds=('IS_FRAUD', 'sum'),
          fraud_rate=('IS_FRAUD', 'mean')
      )
      .reset_index()
)

risk_summary



In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(
    data=risk_summary,
    x='risk_tier',
    y='fraud_rate'
)
plt.title("Fraud Rate by Risk Tier")
plt.ylabel("Fraud Rate")
plt.xlabel("Risk Tier")
plt.show()


Fraud rates are highest within the Medium-Risk tier, while the Low-Risk tier exhibits materially lower fraud incidence. The absence of observed fraud in the High-Risk tier likely reflects low sample volume rather than reduced risk, underscoring the importance of validating tier thresholds and distribution.

Behavioral Baseline Summary

Fraud is rare overall, but risk is highly concentrated by amount and time of day

High-value and overnight transactions represent disproportionate risk

Tiered risk signals enable focused review without broad customer friction

These insights form the foundation for explainable anomaly detection and stakeholder KPIs.

KPI COMPUTATION

In [ ]:
kpis = (
    df.groupby('risk_tier')
      .agg(
          transactions=('AMOUNT', 'count'),
          frauds=('IS_FRAUD', 'sum'),
          fraud_rate=('IS_FRAUD', 'mean'),
          avg_amount=('AMOUNT', 'mean')
      )
      .reset_index()
)


Validate Risk Coverage vs Noise

In [ ]:
risk_eval = (
    df.groupby('risk_tier')
      .agg(
          transactions=('AMOUNT', 'count'),
          frauds=('IS_FRAUD', 'sum'),
          fraud_rate=('IS_FRAUD', 'mean'),
          fraud_coverage=('IS_FRAUD', 'sum')
      )
      .reset_index()
)

total_frauds = df['IS_FRAUD'].sum()
risk_eval['fraud_coverage_pct'] = risk_eval['fraud_coverage'] / total_frauds

risk_eval


In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(
    data=risk_eval,
    x='risk_tier',
    y='fraud_coverage_pct'
)
plt.title("Fraud Coverage by Risk Tier")
plt.ylabel("Share of Total Fraud")
plt.xlabel("Risk Tier")
plt.show()


The majority of fraudulent transactions fall within the Low-Risk tier, indicating that current rule-based thresholds are conservative and prioritize minimizing customer friction over aggressive fraud capture. This highlights an opportunity to refine risk tier definitions to improve fraud concentration without materially increasing false positives.

Statistical Anomaly Signal

In [ ]:
amount_mean = df['AMOUNT'].mean()
amount_std = df['AMOUNT'].std()

df['AMOUNT_Z_SCORE'] = (
    (df['AMOUNT'] - amount_mean) / amount_std
)

df['AMOUNT_ANOMALY_FLAG'] = df['AMOUNT_Z_SCORE'].abs() > 3


Combine Signals

In [ ]:
df['FINAL_RISK_FLAG'] = (
    (df['risk_tier'] == 'HIGH') |
    (df['AMOUNT_ANOMALY_FLAG'])
)


Evaluating Operational Impact

In [ ]:
operational_metrics = {
    'total_transactions': len(df),
    'flagged_transactions': df['FINAL_RISK_FLAG'].sum(),
    'flag_rate': df['FINAL_RISK_FLAG'].mean(),
    'frauds_captured': df[df['FINAL_RISK_FLAG']]['IS_FRAUD'].sum(),
    'fraud_recall': (
        df[df['FINAL_RISK_FLAG']]['IS_FRAUD'].sum() /
        df['IS_FRAUD'].sum()
    )
}

operational_metrics


In [ ]:
df['AMOUNT_BUCKET'] = df['AMOUNT_BUCKET'].astype(str)




In [ ]:
df.columns = [c.upper() for c in df.columns]


In [ ]:
session.write_pandas(
    df,
    table_name="TRANSACTIONS_RISK",
    schema="ANALYSIS",
    overwrite=True
)


In [ ]:
SELECT COUNT(*) FROM ANALYSIS.TRANSACTIONS_RISK;


Fraud Operations KPI

In [ ]:
total_txns = len(df)
flagged_txns = df['FINAL_RISK_FLAG'].sum()
frauds_total = df['IS_FRAUD'].sum()
frauds_captured = df[df['FINAL_RISK_FLAG']]['IS_FRAUD'].sum()

ops_kpis = {
    "total_transactions": total_txns,
    "flagged_transactions": flagged_txns,
    "flag_rate_pct": flagged_txns / total_txns,
    "fraud_recall_pct": frauds_captured / frauds_total
}

ops_kpis


Operations Insight:
Only a small fraction of transactions (1.4%) are flagged for review, significantly reducing operational queue volume. However, the current configuration captures a limited share of total fraud (2.2%), indicating a conservative threshold that prioritizes customer experience over fraud recall and presents an opportunity for further tuning.

RISK ANALYTICS KPI

In [ ]:
risk_kpis = (
    df.groupby('RISK_TIER')
      .agg(
          transactions=('AMOUNT', 'count'),
          frauds=('IS_FRAUD', 'sum'),
          fraud_rate=('IS_FRAUD', 'mean')
      )
      .reset_index()
)

risk_kpis


In [ ]:
risk_kpis['fraud_coverage_pct'] = (
    risk_kpis['frauds'] / frauds_total
)

risk_kpis


Risk Insight:
The majority of fraudulent transactions fall within the Low-Risk tier, indicating that current risk thresholds are highly conservative. While this approach minimizes customer friction, it does not effectively concentrate fraud into higher tiers, highlighting a clear opportunity to refine risk segmentation to improve fraud capture without disproportionately increasing review volume.

Finance Impact KPIs

In [ ]:
AVG_FRAUD_LOSS = 250        # conservative industry estimate
MANUAL_REVIEW_COST = 15    # ops cost per review

estimated_loss_prevented = frauds_captured * AVG_FRAUD_LOSS
review_cost = flagged_txns * MANUAL_REVIEW_COST
net_benefit = estimated_loss_prevented - review_cost

finance_kpis = {
    "estimated_loss_prevented": estimated_loss_prevented,
    "review_cost": review_cost,
    "net_benefit": net_benefit
}

finance_kpis


Finance Insight:
While targeted reviews prevent a small amount of fraud loss, the associated operational review costs significantly exceed the financial benefit under the current configuration, resulting in a negative net impact. This highlights the need to refine review thresholds and prioritization to improve economic efficiency before scaling.

Product / Customer Friction Metrics

In [ ]:
customer_friction_rate = flagged_txns / total_txns

customer_friction_rate


Product Insight:
Customer friction remains low, with only 1.4% of transactions subject to additional controls. This indicates that safeguards are applied selectively, minimizing disruption for the vast majority of customers, while leaving room to further refine targeting toward higher-risk behavior.

In [ ]:
exec_kpi_summary = pd.DataFrame({
    "Metric": [
        "Fraud Recall",
        "Flag Rate",
        "Estimated Loss Prevented ($)",
        "Manual Review Cost ($)",
        "Net Benefit ($)"
    ],
    "Value": [
        f"{ops_kpis['fraud_recall_pct']:.2%}",
        f"{ops_kpis['flag_rate_pct']:.2%}",
        f"${finance_kpis['estimated_loss_prevented']:,.0f}",
        f"${finance_kpis['review_cost']:,.0f}",
        f"${finance_kpis['net_benefit']:,.0f}"
    ]
})

exec_kpi_summary


CREATE VIEWS

In [ ]:
session.write_pandas(
    risk_kpis,
    table_name="RISK_KPIS",
    schema="ANALYSIS",
    overwrite=True
)

session.write_pandas(
    exec_kpi_summary,
    table_name="EXEC_KPI_SUMMARY",
    schema="ANALYSIS",
    overwrite=True
)


In [ ]:
CREATE OR REPLACE VIEW ANALYSIS.RISK_TIME_KPIS AS
SELECT
    DATE(transaction_time) AS date,
    risk_tier,
    COUNT(*) AS transactions,
    SUM(is_fraud) AS frauds,
    AVG(is_fraud) AS fraud_rate
FROM ANALYSIS.TRANSACTIONS_RISK
GROUP BY 1,2;


In [ ]:
CREATE OR REPLACE VIEW ANALYSIS.RISK_DISTRIBUTION AS
SELECT
    amount_bucket,
    risk_tier,
    COUNT(*) AS transactions,
    SUM(is_fraud) AS frauds,
    AVG(is_fraud) AS fraud_rate
FROM ANALYSIS.TRANSACTIONS_RISK
GROUP BY 1,2;


## CUSTOMER-LEVEL RISK CONTEXT

In [ ]:
np.random.seed(42)

n_customers = 10000  # reasonable population size
df['customer_id'] = np.random.randint(1, n_customers + 1, size=len(df))


In [ ]:
df['customer_avg_amount'] = (
    df.groupby('customer_id')['AMOUNT']
      .transform('mean')
)

df['amount_deviation_ratio'] = df['AMOUNT'] / df['customer_avg_amount']


In [ ]:
df['transaction_ts'] = pd.to_datetime(
    df['TRANSACTION_TIME'],
    unit='s',
    origin='unix'
)




In [ ]:
df_sorted = df.sort_values(['customer_id', 'transaction_ts'])

txn_velocity = (
    df_sorted
    .groupby('customer_id')
    .rolling('24h', on='transaction_ts')['AMOUNT']
    .count()
    .reset_index()
    .rename(columns={'AMOUNT': 'txn_count_24h'})
)






In [ ]:
df = df.merge(
    txn_velocity,
    on=['customer_id', 'transaction_ts'],
    how='left'
)


In [ ]:
df['txn_count_24h'] = df['txn_count_24h'].fillna(1)


In [ ]:
sns.boxplot(
    x='IS_FRAUD',
    y='amount_deviation_ratio',
    data=df[df['amount_deviation_ratio'] < 10]
)
plt.title("Transaction Deviation vs Fraud")
plt.show()


Fraudulent transactions exhibit significantly higher variability in transaction amount relative to a customer’s typical spending behavior. While most non-fraud transactions cluster close to a customer’s average amount, fraudulent transactions show a heavier right tail, indicating large positive deviations from normal behavior.

In [ ]:
df['review_control'] = df['RISK_TIER'].isin(['HIGH'])


In [ ]:
df['review_treatment'] = (
    (df['RISK_TIER'] == 'HEDIUM') |
    ((df['RISK_TIER'] == 'LOW') & (df['amount_deviation_ratio'] > 2))
)


In [ ]:
def compute_metrics(review_flag):
    flagged = df[review_flag]
    fraud_captured = flagged['IS_FRAUD'].sum()
    total_fraud = df['IS_FRAUD'].sum()
    
    flag_rate = len(flagged) / len(df)
    recall = fraud_captured / total_fraud
    
    review_cost = len(flagged) * 15
    loss_prevented = fraud_captured * 250
    net_impact = loss_prevented - review_cost
    
    return pd.Series({
        'Flag Rate': flag_rate,
        'Fraud Recall': recall,
        'Review Cost ($)': review_cost,
        'Loss Prevented ($)': loss_prevented,
        'Net Impact ($)': net_impact
    })

results = pd.DataFrame({
    'Control': compute_metrics(df['review_control']),
    'Treatment': compute_metrics(df['review_treatment'])
})

results


The expanded review strategy significantly increased the review rate and improved fraud recall; however, the additional operational review cost substantially exceeded the incremental fraud loss prevented, resulting in a negative net impact. This indicates that while behavioral escalation improves detection, the current thresholds are too aggressive and require further tuning before production rollout.